# This is to compare the results of Monorail with San Donato Experimental Data

### Load the data comparing the "First_phase" Feature Variation using different cut off and window size

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import IsolationForest, RandomForestClassifier
from sklearn.svm import OneClassSVM
from sklearn.neighbors import LocalOutlierFactor
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBClassifier
from imblearn.over_sampling import SMOTE
from sklearn.impute import SimpleImputer
from sklearn.metrics import (classification_report, confusion_matrix, roc_auc_score, 
                             precision_recall_curve, roc_curve, f1_score, precision_score, recall_score)
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

In [ ]:
Test_10hz_win5 = pd.read_csv('Tfinal_10hz_win5.csv')
Test_10hz_win8 = pd.read_csv('Tfinal_10hz_win8.csv')
Test_10hz_win10 = pd.read_csv('Tfinal_10hz_win10.csv')
Test_10hz_win15 = pd.read_csv('Tfinal_10hz_win15.csv')
Test_15hz_win5 = pd.read_csv('Tfinal_15hz_win5.csv')

In [ ]:
Test_10hz_win10.head()

In [ ]:
print(Test_10hz_win10.columns.tolist())

In [ ]:
first_phase_columns = [col for col in Test_10hz_win10.columns if col.startswith('First_phase')]
print(first_phase_columns)

In [ ]:
import matplotlib.pyplot as plt

def plot_first_phase_boxplots(df, title='Baseline: First_phase_* Features'):
    """
    Plots boxplots of all columns in the DataFrame that start with 'First_phase_'.

    Parameters:
    - df: pandas DataFrame
    - title: Title of the plot
    """
    # Filter columns that start with 'First_phase_'
    first_phase_cols = [col for col in df.columns if col.startswith('First_phase_')]
    
    if not first_phase_cols:
        print("No 'First_phase_' columns found.")
        return

    # Extract relevant data
    data_to_plot = df[first_phase_cols]

    # Plot
    plt.figure(figsize=(12, 6))
    data_to_plot.boxplot(rot=45)
    plt.title(title)
    plt.ylabel('Value')
    plt.tight_layout()
    plt.grid(True)
    plt.show()  

In [ ]:
plot_first_phase_boxplots(Test_10hz_win10)

In [ ]:
import matplotlib.pyplot as plt

def compare_first_phase_with_1hz(df, feature_pairs, title_prefix='Comparison:'):
    """
    Plots side-by-side boxplots comparing normal and 1Hz versions of each feature.

    Parameters:
    - df: pandas DataFrame
    - feature_pairs: list of tuples [(normal_feature, 1hz_feature), ...]
    - title_prefix: prefix for each subplot title
    """
    for normal, hz1 in feature_pairs:
        if normal not in df.columns or hz1 not in df.columns:
            print(f"Missing columns: {normal}, {hz1}")
            continue

        plt.figure(figsize=(6, 4))
        plt.boxplot([df[normal].dropna(), df[hz1].dropna()], labels=[normal, hz1])
        plt.title(f"{title_prefix} {normal} vs {hz1}")
        plt.ylabel('Value')
        plt.grid(True)
        plt.tight_layout()
        plt.show()
        
selected_features = [
    'First_phase_half_time_ratio',
    'First_phase_mean_curvature',
    'First_phase_inflection_point',
    'First_phase_power',
    'First_phase_half_time_ratio_1hz',
    'First_phase_mean_curvature_1hz',
    'First_phase_inflection_point_1hz',
    'First_phase_power_1hz'
]

feature_pairs = [
    ('First_phase_half_time_ratio', 'First_phase_half_time_ratio_1hz'),
    ('First_phase_mean_curvature', 'First_phase_mean_curvature_1hz'),
    ('First_phase_power', 'First_phase_power_1hz')
]

compare_first_phase_with_1hz(Test_10hz_win10, feature_pairs)

In [ ]:
import matplotlib.pyplot as plt

def compare_first_phase_across_datasets(datasets, dataset_labels, features, title_prefix='Comparison Across Datasets:'):
    """
    Plots boxplots comparing the same feature across multiple datasets.

    Parameters:
    - datasets: list of pandas DataFrames
    - dataset_labels: list of labels corresponding to each DataFrame
    - features: list of feature names to compare
    - title_prefix: prefix for each plot title
    """
    for feature in features:
        plt.figure(figsize=(8, 5))
        data = [df[feature].dropna() if feature in df.columns else [] for df in datasets]
        plt.boxplot(data, labels=dataset_labels)
        plt.title(f"{title_prefix} {feature}")
        plt.ylabel('Value')
        plt.grid(True)
        plt.tight_layout()
        plt.show()

In [ ]:
datasets = [Test_10hz_win5, Test_10hz_win8, Test_10hz_win10, Test_10hz_win15, Test_15hz_win5]
labels = ['10Hz Win5', '10Hz Win8', '10Hz Win10', '10Hz Win15', '15Hz Win5']

features_to_compare = [
    'First_phase_half_time_ratio',
    'First_phase_mean_curvature',
    'First_phase_power'
]

compare_first_phase_across_datasets(datasets, labels, features_to_compare)

In [ ]:
import matplotlib.pyplot as plt

def plot_feature_with_1hz_baseline(baseline_df, other_dfs, other_labels, feature_list, baseline_label='10Hz Win10'):
    """
    For each feature, plots a single figure with:
    - baseline normal and 1Hz version
    - normal version from other datasets

    Parameters:
    - baseline_df: DataFrame with both normal and *_1hz features
    - other_dfs: list of DataFrames with only normal features
    - other_labels: list of labels for other datasets
    - feature_list: list of base feature names (without *_1hz)
    - baseline_label: label for the baseline dataset
    """
    for feature in feature_list:
        labels = [f'{baseline_label}'] + other_labels + [f'{baseline_label}_1hz']
        data = []

        # Baseline normal
        data.append(baseline_df[feature].dropna() if feature in baseline_df.columns else [])

        # Other datasets normal
        for df in other_dfs:
            data.append(df[feature].dropna() if feature in df.columns else [])

        # Baseline 1Hz
        hz1_feature = feature + '_1hz'
        data.append(baseline_df[hz1_feature].dropna() if hz1_feature in baseline_df.columns else [])

        # Plot
        plt.figure(figsize=(10, 5))
        plt.boxplot(data, labels=labels)
        plt.title(f'Feature Comparison: {feature} (Baseline + 1Hz)')
        plt.ylabel('Value')
        plt.grid(True)
        plt.tight_layout()
        plt.show()

In [ ]:
other_datasets = [Test_10hz_win5, Test_10hz_win8, Test_10hz_win15, Test_15hz_win5]
other_labels = ['10Hz Win5', '10Hz Win8', '10Hz Win15', '15Hz Win5']

features = [
    'First_phase_half_time_ratio',
    'First_phase_mean_curvature',
    'First_phase_power'
]

plot_feature_with_1hz_baseline(Test_10hz_win10, other_datasets, other_labels, features)

In [ ]:
import matplotlib.pyplot as plt

def plot_feature_by_bc_id(baseline_df, other_dfs, other_labels, features, bc_column='BC_ID', baseline_label='10Hz Win10'):
    """
    For each feature and each BC_ID, plots a single figure comparing:
    - baseline normal and *_1hz
    - normal from other datasets

    Parameters:
    - baseline_df: DataFrame with both normal and *_1hz features
    - other_dfs: list of DataFrames with only normal features
    - other_labels: list of labels for other datasets
    - features: list of base feature names (without *_1hz)
    - bc_column: column name for BC_ID
    - baseline_label: label for baseline dataset
    """
    unique_bc_ids = baseline_df[bc_column].dropna().unique()

    for bc_id in unique_bc_ids:
        # Filter baseline and others by BC_ID
        base_filtered = baseline_df[baseline_df[bc_column] == bc_id]
        others_filtered = [df[df[bc_column] == bc_id] for df in other_dfs]

        for feature in features:
            labels = [f'{baseline_label}'] + other_labels + [f'{baseline_label}_1hz']
            data = []

            # Baseline normal
            data.append(base_filtered[feature].dropna() if feature in base_filtered.columns else [])

            # Other datasets normal
            for df in others_filtered:
                data.append(df[feature].dropna() if feature in df.columns else [])

            # Baseline 1Hz
            hz1_feature = feature + '_1hz'
            data.append(base_filtered[hz1_feature].dropna() if hz1_feature in base_filtered.columns else [])

            # Plot
            plt.figure(figsize=(10, 5))
            plt.boxplot(data, labels=labels)
            plt.title(f'Feature: {feature} | BC_ID: {bc_id}')
            plt.ylabel('Value')
            plt.grid(True)
            plt.tight_layout()
            plt.show()

In [ ]:
other_datasets = [Test_10hz_win5, Test_10hz_win8, Test_10hz_win15, Test_15hz_win5]
other_labels = ['10Hz Win5', '10Hz Win8', '10Hz Win15', '15Hz Win5']

features = [
    'First_phase_half_time_ratio',
    'First_phase_mean_curvature',
    'First_phase_power'
]

plot_feature_by_bc_id(Test_10hz_win10, other_datasets, other_labels, features)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

def plot_feature_with_1hz_baseline_and_median_error(
    baseline_df, other_dfs, other_labels, feature_list, baseline_label='10Hz Win10'
):
    """
    For each feature, plots a single figure with:
    - baseline normal and 1Hz version
    - normal version from other datasets
    Also computes squared error of medians relative to baseline.

    Returns:
    - Dictionary of median squared errors per feature
    """
    median_errors = {}

    for feature in feature_list:
        labels = [f'{baseline_label}'] + other_labels + [f'{baseline_label}_1hz']
        data = []

        # Collect data
        base_normal = baseline_df[feature].dropna() if feature in baseline_df.columns else pd.Series(dtype=float)
        data.append(base_normal)

        for df in other_dfs:
            data.append(df[feature].dropna() if feature in df.columns else pd.Series(dtype=float))

        hz1_feature = feature + '_1hz'
        base_1hz = baseline_df[hz1_feature].dropna() if hz1_feature in baseline_df.columns else pd.Series(dtype=float)
        data.append(base_1hz)

        # Plot
        plt.figure(figsize=(10, 5))
        plt.boxplot(data, labels=labels)
        plt.title(f'Feature Comparison: {feature} (Baseline + 1Hz)')
        plt.ylabel('Value')
        plt.grid(True)
        plt.tight_layout()
        plt.show()

        # Compute median errors
        base_median = np.median(base_normal) if len(base_normal) > 0 else np.nan
        medians = [np.median(d) if len(d) > 0 else np.nan for d in data]
        squared_errors = [(m - base_median) ** 2 if not np.isnan(m) else np.nan for m in medians]

        median_errors[feature] = dict(zip(labels, squared_errors))

    return median_errors

In [ ]:
errors = plot_feature_with_1hz_baseline_and_median_error(
    Test_10hz_win10,
    [Test_10hz_win5, Test_10hz_win8, Test_10hz_win15, Test_15hz_win5],
    ['10Hz Win5', '10Hz Win8', '10Hz Win15', '15Hz Win5'],
    [
        'First_phase_half_time_ratio',
        'First_phase_mean_curvature',
        'First_phase_power'
    ]
)

# Optional: print or inspect
import pprint
pprint.pprint(errors)

In [ ]:
import pandas as pd

# Your existing dictionary (replace this with your actual variable)
median_errors = errors

# Convert to DataFrame
df_errors = pd.DataFrame(median_errors).T  # Transpose to have features as rows
df_errors = df_errors.round(6)  # Optional: round for readability

# Display
print(df_errors)

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# Convert the nested dictionary to a DataFrame
df_errors = pd.DataFrame(median_errors).T.reset_index().rename(columns={'index': 'Feature'})

# Melt to long format for seaborn
df_long = df_errors.melt(id_vars='Feature', var_name='Dataset', value_name='SquaredError')

# Plot
plt.figure(figsize=(12, 6))
sns.barplot(data=df_long, x='Feature', y='SquaredError', hue='Dataset')
plt.title('Median Squared Error Compared to Baseline (10Hz Win10)')
plt.ylabel('Squared Error')
plt.xticks(rotation=45)
plt.grid(True, axis='y')
plt.tight_layout()
plt.legend(title='Dataset')
plt.show()